### Stacking algorithm - regression + CatBoost

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers


In [2]:
# load the diamonds dataset, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_regression_diamonds.csv")
df.head()

,carat,cut,color,clarity,table,price,x,y,z
0,0.29,4,2,4,58.0,334,4.20,4.23,2.63
1,0.26,3,3,3,55.0,337,4.07,4.11,2.53
2,0.30,2,1,3,55.0,339,4.25,4.28,2.73
3,0.31,5,1,2,54.0,344,4.35,4.37,2.71
4,0.32,4,6,1,58.0,345,4.38,4.42,2.68


In [3]:
# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["cut", "color", "clarity"]

# continuous variables also into a list
continuous_variables = ["carat", "table", "price", "depth", "x", "y", "z"]

# the usual X/y -split
X = df.drop("price", axis=1)
y = df['price']

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Stacking algorithms - regression example

In [4]:
from sklearn.ensemble import StackingRegressor

# meta model is the "judge" or the optimizer of the stacking algorithm
# => it assigns weights to the models based on their performance
meta_model = LinearRegression()

# stacking regression algorithm
stacking_model = StackingRegressor(
    estimators=[('xgb', xgb.XGBRegressor(enable_categorical=True, objective="reg:squarederror")),
                ('randomforest', RandomForestRegressor()),
                ('catboost', cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0)),
                ('lgbm', lgb.LGBMRegressor(objective="regression"))],
    final_estimator=meta_model,
    cv=5
)

# other than that, the stacking algorithm works as any other algorithm
stacking_model.fit(X_train, y_train)

# test predictions and metrics
predictions = stacking_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R2: {r2}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000185 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 939
[LightGBM] [Info] Number of data points in the train set: 33375, number of used features: 8
[LightGBM] [Info] Start training from score 2275.175431
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000337 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 933
[LightGBM] [Info] Number of data points in the train set: 26700, number of used features: 8
[LightGBM] [Info] Start training from score 2276.590599
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000843 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 931
[LightGBM] [Info] Number of data points in the train se

In [ ]:
# pretty decent result
# original MAE with Random Forest => 141.9
# original RMSE with CatBoost => 232.67
# original R2 with CatBoost => 0.9825

# the result is slightly better when catboost is added 
# into the mix

# also, stacking algorithm uses cross validation, so these are 
# very good metrics since cross validation always tends to decrease metrics